# 02 — Data Cleaning & Implied Probability Pipeline

**Football Odds Intelligence & Match Analytics**

---

## Objectives
1. Ingest raw matches and multi-bookmaker closing odds.
2. Enforce schema constraints, timestamp parsing, and outcome derivations.
3. Audit and sanitize decimal odds values ($odds > 1.0$).
4. Calculate mathematical betting indicators:
   - Raw implied probability: $P_{implied} = 1 / \text{odds}$
   - Bookmaker overround (margin): $\text{Overround} = \sum P_{implied} - 1$
   - Normalized fair probability: $P_{fair} = P_{implied} / (1 + \text{Overround})$
   - Odds drift: $\Delta\% = \frac{\text{close} - \text{open}}{\text{open}} \times 100$
5. Persist clean datasets to `data/processed/` in Parquet format.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_matches, load_match_closing, load_processed
from src.data.cleaner import clean_matches, clean_match_closing, run_cleaning_pipeline

pd.set_option('display.max_columns', 35)
print('Data cleaner imported successfully!')

## 1. Execute Pipeline

In [ ]:
matches_clean, closing_clean = run_cleaning_pipeline()
print(f"matches_clean shape: {matches_clean.shape}")
print(f"closing_clean shape: {closing_clean.shape}")

## 2. Inspect Cleaned Matches Dataset

In [ ]:
matches_clean[['match_id', 'kickoff_utc', 'league_name', 'home_team', 'away_team', 'ft_home', 'ft_away', 'outcome', 'total_goals']].head(10)

## 3. Inspect Cleaned Closing Odds with Implied Probabilities & Drift

In [ ]:
cols_to_view = [
    'match_id', 'bookmaker', 
    'close_1x2_home', 'close_1x2_draw', 'close_1x2_away',
    'close_overround', 
    'close_norm_prob_home', 'close_norm_prob_draw', 'close_norm_prob_away',
    'drift_home_pct', 'close_favourite'
]
closing_clean[cols_to_view].head(10)

## 4. Verification Check: Normalized Probabilities Conservation

In [ ]:
prob_sums = (
    closing_clean['close_norm_prob_home'] + 
    closing_clean['close_norm_prob_draw'] + 
    closing_clean['close_norm_prob_away']
).dropna()

print(f"Min sum of normalized probs: {prob_sums.min():.6f}")
print(f"Max sum of normalized probs: {prob_sums.max():.6f}")
print(f"Mean sum: {prob_sums.mean():.6f}")
assert np.allclose(prob_sums, 1.0, atol=1e-5), 'Normalized probabilities must sum to 1.0!'